# Exploring eBay Car Sales Data

In this project, I will work with a dataset of used cars from eBay Kleinanzeigen, a classifieds section of the German eBay website. The goal of this project is to clean the dataset and perform some initial analysis using pandas and NumPy.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
autos = pd.read_csv("autos.csv", encoding="Latin-1")

In [ ]:
autos

In [ ]:
autos.info()

In [ ]:
autos.head()

## Initial Observations

The dataset contains 50,000 used car listings and 20 columns. Some columns contain missing values. The column names use camelCase naming, and some columns may need to be cleaned and converted to more appropriate data types before analysis.

In [ ]:
print(autos.columns)

In [ ]:
autos.columns = [
    "date_crawled",
    "name",
    "seller",
    "offer_type",
    "price",
    "abtest",
    "vehicle_type",
    "registration_year",
    "gearbox",
    "power_ps",
    "model",
    "odometer",
    "registration_month",
    "fuel_type",
    "brand",
    "unrepaired_damage",
    "ad_created",
    "nr_of_pictures",
    "postal_code",
    "last_seen"
]

In [ ]:
autos.head()

## Cleaning Column Names

I changed the column names from camelCase to snake_case to make them easier to read and work with in Python. I also renamed some columns, such as `yearOfRegistration` to `registration_year` and `notRepairedDamage` to `unrepaired_damage`, to make their meanings clearer.

In [ ]:
autos.describe(include="all")

In [ ]:
print(autos["price"].head())
print(autos["odometer"].head())

In [ ]:
autos["price"] = (
    autos["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(int)
)

In [ ]:
autos["odometer"] = (
    autos["odometer"]
    .str.replace(",", "", regex=False)
    .str.replace("km", "", regex=False)
    .astype(int)
)

In [ ]:
autos.rename(
    {"odometer": "odometer_km"},
    axis=1,
    inplace=True
)

In [ ]:
print(autos[["price", "odometer_km"]].head())
print(autos[["price", "odometer_km"]].dtypes)

## Initial Exploration and Cleaning

After exploring the dataset, I found that some columns contain mostly the same value and may not be useful for analysis. In particular, `seller` and `offer_type` are candidates to be dropped.

The `price` and `odometer` columns contained numeric information stored as text. I removed the non-numeric characters and converted these columns to numeric data types. I also renamed `odometer` to `odometer_km` to make the unit of measurement clear.

Some columns, such as `registration_year`, may require further investigation because they could contain unrealistic values.

In [ ]:
print(autos["odometer_km"].unique().shape)
print(autos["odometer_km"].describe())
print(autos["odometer_km"].value_counts().sort_index())

In [ ]:
print(autos["price"].unique().shape)
print(autos["price"].describe())

In [ ]:
autos = autos[autos["price"].between(1, 350000)]

In [ ]:
autos["price"].between(1, 350000)

In [ ]:
print(autos["price"].describe())
print(autos["odometer_km"].describe())

In [ ]:
autos.shape

## Exploring Price and Odometer

After examining the `odometer_km` column, I found that the values are rounded and there are no obvious unrealistic outliers that need to be removed.

The `price` column contains some unrealistic values. Some vehicles have a price of 0, while others have extremely high prices that are unlikely to represent genuine used-car listings. I removed listings with prices below €1 and above €350,000.

After removing these outliers, the price distribution is more suitable for further analysis.

In [ ]:
autos["date_crawled"].str[:10].value_counts(
    normalize=True,
    dropna=False
).sort_index()

### Date Crawled

The `date_crawled` values cover approximately one month, from March to April 2016. The crawling activity appears to be fairly evenly distributed across this period.

In [ ]:
autos["ad_created"].str[:10].value_counts(
    normalize=True,
    dropna=False
).sort_index()

### Ad Created

The `ad_created` dates cover a wider range than the crawling dates. Most advertisements were created close to the crawling period, although some listings were created earlier.

In [ ]:
autos["last_seen"].str[:10].value_counts(
    normalize=True,
    dropna=False
).sort_index()

### Last Seen

The `last_seen` dates are concentrated between March and April 2016. A noticeable proportion of listings were last seen during the final days of the crawling period.

In [ ]:
autos["registration_year"].describe()

### Registration Year

The `registration_year` column contains some unrealistic values. The minimum and maximum registration years indicate that there are invalid entries in the dataset. These values will need further investigation and cleaning before analyzing vehicle registration years.

### Cleaning Registration Year

Since the listings were crawled in 2016, a vehicle cannot have a registration year later than 2016. I will use 1900 as the lower limit because cars registered before 1900 are extremely unlikely in this used-car dataset. Therefore, I will keep registration years between 1900 and 2016.

In [ ]:
autos = autos[autos["registration_year"].between(1900, 2016)]

In [ ]:
autos["registration_year"].value_counts(normalize=True)

### Registration Year Distribution

After removing registration years outside the 1900–2016 range, the remaining values appear more realistic. Most of the cars were registered in the 1990s and 2000s, with fewer vehicles from the earlier decades.

In [ ]:
brand_counts = autos["brand"].value_counts(normalize=True)
print(brand_counts)

In [ ]:
common_brands = brand_counts[brand_counts > 0.05].index

print(common_brands)

### Brand Selection

The `brand` column contains many different car brands. Volkswagen is the most common brand in the dataset. To focus the analysis on brands with a significant number of listings, I selected brands that represent more than 5% of the total listings.

In [ ]:
brand_mean_prices = {}

for brand in common_brands:
    brand_only = autos[autos["brand"] == brand]
    mean_price = brand_only["price"].mean()
    brand_mean_prices[brand] = mean_price

print(brand_mean_prices)

In [ ]:
mean_price = brand_only["price"].mean()

### Average Price by Brand

There are noticeable differences in average prices between the most common brands. Premium brands such as Audi, BMW, and Mercedes-Benz tend to have higher average prices, while brands such as Opel and Ford tend to have lower average prices. Volkswagen falls somewhere in between.

In [ ]:
brand_mean_prices = {}
brand_mean_mileage = {}

for brand in common_brands:
    brand_only = autos[autos["brand"] == brand]
    
    mean_price = brand_only["price"].mean()
    mean_mileage = brand_only["odometer_km"].mean()
    
    brand_mean_prices[brand] = mean_price
    brand_mean_mileage[brand] = mean_mileage

In [ ]:
bmp_series = pd.Series(brand_mean_prices)
bmm_series = pd.Series(brand_mean_mileage)

print(bmp_series)
print(bmm_series)

In [ ]:
brand_info = pd.DataFrame(
    bmp_series,
    columns=["mean_price"]
)

### Price and Mileage by Brand

The average mileage does not vary as much as the average price among the most common brands. Audi, BMW, and Mercedes-Benz have higher average prices, while Opel and Ford are less expensive. Volkswagen is in the middle of the price range.

The more expensive brands do not necessarily have much lower mileage, suggesting that brand has a strong influence on price in addition to mileage.